In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import os

RESULTS_DIR = Path("/content/drive/MyDrive/Colab Notebooks/results")
DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/data")
MODEL_DIR = RESULTS_DIR / "models"
IMPORTANCE_DIR = RESULTS_DIR / "feature_importance"

results_file = RESULTS_DIR / "model_comparison.csv"

print("Results folder exists:", RESULTS_DIR.exists())
print("Model folder exists:", MODEL_DIR.exists())
print("Feature importance folder exists:", IMPORTANCE_DIR.exists())
print("Model comparison file exists:", results_file.exists())

if results_file.exists():
    results = pd.read_csv(results_file)
    print("\nResults shape:", results.shape)
    print("Column names:", results.columns.tolist())
    display(results.head())
else:
    print("\nCould not find model_comparison.csv.")
    print("Files in results folder:")
    if RESULTS_DIR.exists():
        for file in RESULTS_DIR.iterdir():
            print(file.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results folder exists: True
Model folder exists: True
Feature importance folder exists: True
Model comparison file exists: True

Results shape: (45, 11)
Column names: ['Feature Group', 'Target', 'Model', 'Selected Features', 'Train Rows After SMOTEENN', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'PR-AUC (Average Precision)']


,Feature Group,Target,Model,Selected Features,Train Rows After SMOTEENN,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC (Average Precision)
0,Radiom,ICU,Logistic Regression,113,1170,0.305556,0.296192,0.995261,0.456522,0.716887,0.502241
1,Radiom,ICU,Random Forest,113,1170,0.294444,0.293463,1.000000,0.453763,0.630620,0.368215
2,Radiom,ICU,SVM,113,1170,0.302778,0.295933,1.000000,0.456710,0.632231,0.369943
3,Radiom,ICU,MLP,113,1170,0.298611,0.294693,1.000000,0.455232,0.615136,0.423261
4,Radiom,ICU,LightGBM,113,1170,0.305556,0.296765,1.000000,0.457701,0.644215,0.377242


In [2]:
# Rank models separately for each target using ROC-AUC
ranked_results = results.sort_values(
    ["Target", "ROC-AUC"],
    ascending=[True, False]
).copy()

ranked_results["Rank"] = (
    ranked_results.groupby("Target")["ROC-AUC"]
    .rank(method="min", ascending=False)
    .astype(int)
)

columns_to_show = [
    "Target",
    "Feature Group",
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC",
    "PR-AUC (Average Precision)",
    "Rank"
]

display(
    ranked_results[columns_to_show]
    .round(3)
    .sort_values(["Target", "Rank"])
    .reset_index(drop=True)
)

,Target,Feature Group,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC (Average Precision),Rank
0,ICU,RadioClinLab,Logistic Regression,0.321,0.301,1.000,0.463,0.721,0.509,1
1,ICU,Radiom,Logistic Regression,0.306,0.296,0.995,0.457,0.717,0.502,2
2,ICU,RadioClinLab,MLP,0.315,0.300,1.000,0.461,0.714,0.493,3
3,ICU,RadioClin,Logistic Regression,0.308,0.298,1.000,0.459,0.684,0.460,4
4,ICU,RadioClinLab,Random Forest,0.301,0.296,1.000,0.456,0.681,0.409,5
5,ICU,RadioClinLab,LightGBM,0.336,0.306,1.000,0.469,0.670,0.389,6
6,ICU,Radiom,LightGBM,0.306,0.297,1.000,0.458,0.644,0.377,7
7,ICU,Radiom,SVM,0.303,0.296,1.000,0.457,0.632,0.370,8
8,ICU,RadioClin,Random Forest,0.296,0.294,1.000,0.454,0.632,0.365,9
9,ICU,Radiom,Random Forest,0.294,0.293,1.000,0.454,0.631,0.368,10


In [3]:
best_models = (
    results.sort_values("ROC-AUC", ascending=False)
    .groupby("Target", as_index=False)
    .head(1)
    .sort_values("Target")
    .reset_index(drop=True)
)

display(
    best_models[
        [
            "Target",
            "Feature Group",
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1 Score",
            "ROC-AUC",
            "PR-AUC (Average Precision)"
        ]
    ].round(3)
)

,Target,Feature Group,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC (Average Precision)
0,ICU,RadioClinLab,Logistic Regression,0.321,0.301,1.000,0.463,0.721,0.509
1,death,RadioClinLab,Random Forest,0.200,0.172,1.000,0.294,0.730,0.317
2,mechanical_ventilation,RadioClinLab,Logistic Regression,0.343,0.308,0.967,0.467,0.737,0.552


In [4]:
ranked_path = RESULTS_DIR / "ranked_model_comparison.csv"
best_path = RESULTS_DIR / "best_models_by_target.csv"

ranked_results.to_csv(ranked_path, index=False)
best_models.to_csv(best_path, index=False)

print("Saved ranked comparison:", ranked_path)
print("Saved best-model summary:", best_path)

Saved ranked comparison: /content/drive/MyDrive/Colab Notebooks/results/ranked_model_comparison.csv
Saved best-model summary: /content/drive/MyDrive/Colab Notebooks/results/best_models_by_target.csv
